# Heart Disease Prediction — Exploratory Data Analysis

**Course:** Machine Learning Operations (MLOps) - AIMLCZG523  
**Assignment:** Assignment 01  
**Dataset:** Heart Disease UCI Dataset  
**Source:** UCI Machine Learning Repository  

## Objective

The objective of this analysis is to explore and prepare the UCI Heart
Disease dataset for development of a binary machine-learning classifier
that predicts the presence or absence of heart disease.

The analysis includes data-quality assessment, missing-value analysis,
target transformation, class-balance analysis, feature distributions,
categorical analysis, correlation analysis, and creation of a processed
dataset for model development.

In [ ]:
import os

print(os.getcwd())

/home/s2025ae05100/workspace/heart-disease-mlops/notebooks


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

FIGURE_DIR = Path("../reports/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("EDA environment ready")

EDA environment ready


## 1. Data Acquisition

The Heart Disease dataset was obtained from the official UCI Machine
Learning Repository (Dataset ID 45).

A reproducible download script is maintained in `src/download_data.py`.
The downloaded source data is stored in `data/raw/heart_disease.csv`,
keeping raw data separate from processed modelling data.

In [2]:
DATA_PATH = Path("../data/raw/heart_disease.csv")

print("Dataset path:", DATA_PATH.resolve())
print("File exists:", DATA_PATH.exists())

df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully.")
print("Dataset shape:", df.shape)

df.head()

Dataset path: /home/s2025ae05100/workspace/heart-disease-mlops/data/raw/heart_disease.csv
File exists: True

Dataset loaded successfully.
Dataset shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


## 2. Dataset Structure and Data Quality

The dataset structure is examined to understand the number of observations,
available attributes, data types, descriptive statistics, duplicate records,
and missing values.

In [3]:
print("Number of observations:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

Number of observations: 303
Number of columns: 14

Column names:
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  num       303 non-null    int64  
dtypes: float64(3), int64(11)
memory usage: 33.3 KB


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,303.0,54.438944,9.038662,29.0,48.0,56.0,61.0,77.0
sex,303.0,0.679868,0.467299,0.0,0.0,1.0,1.0,1.0
cp,303.0,3.158416,0.960126,1.0,3.0,3.0,4.0,4.0
trestbps,303.0,131.689769,17.599748,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.693069,51.776918,126.0,211.0,241.0,275.0,564.0
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.990099,0.994971,0.0,0.0,1.0,2.0,2.0
thalach,303.0,149.607261,22.875003,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2


In [6]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


### Duplicate Analysis

Duplicate observations were checked before preprocessing. No duplicate
records were detected in the dataset, so duplicate removal was not required.

## 3. Missing Value Analysis

Missing values are examined before model development because incomplete
observations can affect model training and production inference.

In [1]:
missing_count = df.isnull().sum()

missing_percentage = (
    df.isnull().mean() * 100
).round(2)

missing_table = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage (%)": missing_percentage
})

missing_table.sort_values(
    by="Missing Count",
    ascending=False
)

NameError: name 'df' is not defined

In [2]:
missing_only = missing_table[
    missing_table["Missing Count"] > 0
]

plt.figure(figsize=(7, 4))

ax = sns.barplot(
    x=missing_only.index,
    y=missing_only["Missing Count"]
)

plt.title("Missing Values by Feature")
plt.xlabel("Feature")
plt.ylabel("Missing Value Count")

for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "missing_values.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

NameError: name 'missing_table' is not defined

### Missing Value Handling Decision

Missing observations were identified in the `ca` and `thal` features.

Since the dataset contains only 303 observations, deleting incomplete
records would unnecessarily reduce the available training data. Missing
values are therefore retained at the EDA stage and will be handled using
imputation within the scikit-learn preprocessing pipeline.

Keeping imputation inside the pipeline also ensures that identical
preprocessing is applied during training and production inference.

## 4. Target Variable Preparation

The assignment requires a binary target representing the presence or absence
of heart disease.

The original UCI Cleveland dataset provides the diagnosis in the `num`
variable. Therefore, the original diagnosis is examined and converted into
a binary target suitable for this assignment.

In [ ]:
original_target_counts = (
    df["num"]
    .value_counts()
    .sort_index()
)

print("Original UCI target distribution:")
print(original_target_counts)

In [ ]:
df["target"] = (df["num"] > 0).astype(int)

print("Binary target created successfully.")

target_mapping = pd.crosstab(
    df["num"],
    df["target"],
    rownames=["Original UCI num"],
    colnames=["Binary target"]
)

target_mapping

### Binary Target Definition

For this assignment, the original UCI diagnosis is converted to binary
classification as follows:

- **Target = 0:** absence of heart disease (`num = 0`)
- **Target = 1:** presence of heart disease (`num > 0`)

The original `num` variable will be removed from the modelling dataset after
this conversion. Since `target` is directly derived from `num`, retaining
`num` as an input feature would introduce target leakage.

In [ ]:
class_counts = (
    df["target"]
    .value_counts()
    .sort_index()
)

class_percentages = (
    df["target"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

class_summary = pd.DataFrame({
    "Count": class_counts,
    "Percentage (%)": class_percentages
})

class_summary.index = [
    "No Heart Disease",
    "Heart Disease Present"
]

class_summary

In [ ]:
plt.figure(figsize=(7, 5))

ax = sns.countplot(
    data=df,
    x="target"
)

plt.title("Heart Disease Class Distribution")
plt.xlabel("Heart Disease Status")
plt.ylabel("Number of Patients")

ax.set_xticks([0, 1])
ax.set_xticklabels([
    "No Heart Disease",
    "Heart Disease"
])

for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "class_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Class Balance Observation

Both diagnostic classes are represented in the dataset without an extreme
class imbalance.

Nevertheless, model performance will not be evaluated using accuracy alone.
Precision, recall, F1-score, and ROC-AUC will also be considered during
model development.

## 5. Feature Analysis

The dataset contains continuous clinical measurements as well as categorical
attributes represented using numeric codes. These feature groups are
identified separately because they require different preprocessing
strategies during model development.

In [ ]:
continuous_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

print("Continuous features:")
print(continuous_features)

print("\nCategorical features:")
print(categorical_features)

print("\nContinuous feature count:", len(continuous_features))
print("Categorical feature count:", len(categorical_features))

### Class Balance Observation

Both diagnostic classes are represented in the dataset without an extreme
class imbalance.

Nevertheless, model performance will not be evaluated using accuracy alone.
Precision, recall, F1-score, and ROC-AUC will also be considered during
model development.

In [ ]:
continuous_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

print("Continuous features:")
print(continuous_features)

print("\nCategorical features:")
print(categorical_features)

print("\nContinuous feature count:", len(continuous_features))
print("Categorical feature count:", len(categorical_features))

In [ ]:
df[continuous_features].hist(
    figsize=(12, 8),
    bins=20,
    edgecolor="black"
)

plt.suptitle(
    "Distribution of Continuous Health Features",
    fontsize=14
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "continuous_feature_histograms.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=df,
    x="age",
    hue="target",
    bins=15,
    kde=True,
    multiple="layer"
)

plt.title("Age Distribution by Heart Disease Status")
plt.xlabel("Age")
plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "age_by_target.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=df,
    x="chol",
    hue="target",
    bins=20,
    kde=True,
    multiple="layer"
)

plt.title("Cholesterol Distribution by Heart Disease Status")
plt.xlabel("Serum Cholesterol")
plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "cholesterol_by_target.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

sns.boxplot(
    data=df,
    x="target",
    y="thalach"
)

plt.title("Maximum Heart Rate by Heart Disease Status")
plt.xlabel("Heart Disease Status")
plt.ylabel("Maximum Heart Rate Achieved")

plt.xticks(
    [0, 1],
    ["Absent", "Present"]
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "heart_rate_by_target.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Chest Pain Category Analysis

The `cp` variable is represented using numeric codes but is a categorical
clinical attribute. Therefore, its relationship with heart disease is
examined using category-level proportions.

### Chest Pain Category Analysis

The `cp` variable is represented using numeric codes but is a categorical
clinical attribute. Therefore, its relationship with heart disease is
examined using category-level proportions.

In [ ]:
cp_distribution = pd.crosstab(
    df["cp"],
    df["target"],
    normalize="index"
).mul(100)

cp_distribution.round(2)

In [ ]:
cp_distribution.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title(
    "Heart Disease Distribution by Chest Pain Category"
)

plt.xlabel("Chest Pain Category")
plt.ylabel("Percentage of Patients")

plt.legend(
    ["No Heart Disease", "Heart Disease"],
    title="Diagnosis"
)

plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "chest_pain_by_target.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 6. Correlation Analysis

Correlation analysis is used to explore linear associations among the
dataset variables and the binary target.

Correlation indicates statistical association and should not be interpreted
as evidence of causation.

The original `num` variable is excluded from this analysis because the
binary `target` was directly derived from it.

In [ ]:
correlation_df = df.drop(
    columns=["num"]
)

correlation_matrix = correlation_df.corr(
    numeric_only=True
)

plt.figure(figsize=(14, 10))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)

plt.title(
    "Correlation Heatmap of Heart Disease Features"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
target_correlations = (
    correlation_matrix["target"]
    .drop("target")
    .sort_values()
)

target_correlations

In [ ]:
plt.figure(figsize=(8, 6))

target_correlations.plot(
    kind="barh"
)

plt.title(
    "Feature Correlation with Heart Disease Target"
)

plt.xlabel("Correlation Coefficient")
plt.ylabel("Feature")

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "target_correlations.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 7. Processed Dataset Preparation

The final modelling dataset contains the 13 patient predictor variables and
one binary target representing the presence or absence of heart disease.

The original `num` diagnosis is removed to prevent target leakage.

Missing values are intentionally retained at this stage because imputation
will be performed within the machine-learning preprocessing pipeline in the
next stage.

In [ ]:
processed_df = df.drop(
    columns=["num"]
).copy()

print("Processed dataset shape:", processed_df.shape)

print(
    "Number of predictor features:",
    processed_df.drop(columns=["target"]).shape[1]
)

print(
    "Target values:",
    sorted(processed_df["target"].unique())
)

print(
    "Duplicate rows:",
    processed_df.duplicated().sum()
)

print("\nMissing values:")
print(
    processed_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

processed_df.head()

In [ ]:
processed_df.to_csv(
    PROCESSED_PATH,
    index=False
)

print("Processed dataset saved successfully.")
print("Location:", PROCESSED_PATH.resolve())

In [ ]:
verification_df = pd.read_csv(
    PROCESSED_PATH
)

print("Reloaded shape:", verification_df.shape)

print(
    "Binary target values:",
    sorted(verification_df["target"].unique())
)

print(
    "Original num present:",
    "num" in verification_df.columns
)

verification_df.head()

## 8. EDA Summary

The official UCI Heart Disease dataset was successfully acquired
programmatically and retained separately as raw data for reproducibility.

The exploratory analysis included:

- dataset structure and descriptive statistics;
- duplicate-record detection;
- missing-value analysis;
- conversion of the original UCI diagnosis into a binary target;
- class-balance analysis;
- continuous-feature histograms;
- selected feature-versus-target analysis;
- categorical chest-pain analysis; and
- correlation analysis.

The original UCI `num` diagnosis was transformed into a binary target where
0 represents absence of heart disease and 1 represents presence of heart
disease.

Missing observations in `ca` and `thal` were retained because imputation
will be incorporated into the scikit-learn preprocessing pipeline rather
than performed globally before train/test splitting.

The original `num` variable was removed after target construction to prevent
target leakage.

The resulting processed dataset contains 303 observations, 13 predictor
features, and one binary target and is ready for feature engineering and
model development.

In [11]:
print("Current dataframe shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

print("\nTarget exists:", "target" in df.columns)


Current dataframe shape: (303, 14)
Columns:
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']

Target exists: False


In [12]:
df["target"] = (df["num"] > 0).astype(int)

print(df[["num", "target"]].head())
print("\nBinary target distribution:")
print(df["target"].value_counts().sort_index())

   num  target
0    0       0
1    2       1
2    1       1
3    0       0
4    0       0

Binary target distribution:
target
0    164
1    139
Name: count, dtype: int64


In [13]:
processed_df = df.drop(
    columns=["num"]
).copy()

print("Processed dataset shape:", processed_df.shape)

print("\nColumns:")
print(processed_df.columns.tolist())

print("\nTarget values:")
print(sorted(processed_df["target"].unique()))

print("\nMissing values:")
print(
    processed_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

Processed dataset shape: (303, 14)

Columns:
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

Target values:
[np.int64(0), np.int64(1)]

Missing values:
ca          4
thal        2
cp          0
trestbps    0
age         0
sex         0
fbs         0
chol        0
restecg     0
thalach     0
oldpeak     0
exang       0
slope       0
target      0
dtype: int64


In [14]:
from pathlib import Path

PROCESSED_PATH = Path(
    "../data/processed/heart_disease_processed.csv"
)

PROCESSED_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

processed_df.to_csv(
    PROCESSED_PATH,
    index=False
)

print("Processed dataset saved successfully.")
print("Path:", PROCESSED_PATH.resolve())
print("File exists:", PROCESSED_PATH.exists())

Processed dataset saved successfully.
Path: /home/s2025ae05100/workspace/heart-disease-mlops/data/processed/heart_disease_processed.csv
File exists: True


In [15]:
verification_df = pd.read_csv(
    PROCESSED_PATH
)

print("Reloaded successfully.")
print("Shape:", verification_df.shape)
print("Target values:", sorted(verification_df["target"].unique()))
print("'num' present:", "num" in verification_df.columns)

verification_df.head()

Reloaded successfully.
Shape: (303, 14)
Target values: [np.int64(0), np.int64(1)]
'num' present: False


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0
